# Decision Tree Classification – COVID-19 Dataset

This notebook follows the same general workflow as the Iris decision-tree notebook, adapted for the COVID-19 CSV dataset.

**Prediction target used here:** whether the patient died, derived from `DATE_DIED` (`9999-99-99` means the patient did not die; any actual date means the patient died). If your assignment requires predicting a different column, change the target section below.

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
# 2. Load dataset
# Put the CSV file in the same folder as this notebook, then update the filename if needed.
file_path = "covid_dataset.csv"

data = pd.read_csv(file_path)

print("Original Dataset")
display(data.head())
print("\nDataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())


In [ ]:
# 3. Inspect the dataset
display(data.describe(include="all").T)
print("\nMissing values in each column:")
print(data.isnull().sum())


In [ ]:
# 4. Prepare the target
# In this dataset, DATE_DIED = '9999-99-99' indicates that the patient did not die.
# An actual date indicates that the patient died.

data = data.copy()
data["DATE_DIED"] = data["DATE_DIED"].astype(str).str.strip()
data["DIED"] = (data["DATE_DIED"] != "9999-99-99").astype(int)

print("Target counts (0 = survived, 1 = died):")
print(data["DIED"].value_counts())


In [ ]:
# 5. Separate features and target
# DATE_DIED is removed from X because it was used to create the target.
X = data.drop(columns=["DATE_DIED", "DIED"])
y = data["DIED"]

# Convert any non-numeric feature columns to numeric where possible.
# Decision trees in scikit-learn require numeric input.
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())


In [ ]:
# 6. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


In [ ]:
# 7. Handle missing / unknown values
# Some columns use codes such as 97, 98, or 99 for unknown/not applicable.
# Replace these codes with NaN in columns where they occur, then impute missing
# values using the median calculated from the training data only.
unknown_codes = [97, 98, 99]

X_train = X_train.copy()
X_test = X_test.copy()

for col in X.columns:
    X_train.loc[X_train[col].isin(unknown_codes), col] = np.nan
    X_test.loc[X_test[col].isin(unknown_codes), col] = np.nan

imputer = SimpleImputer(strategy="median")
X_train_ready = imputer.fit_transform(X_train)
X_test_ready = imputer.transform(X_test)

print("Preprocessing complete.")
print("First five processed training rows:")
print(X_train_ready[:5])


In [ ]:
# 8. Create and train the Decision Tree
dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    random_state=42,
    class_weight="balanced"
)

dt.fit(X_train_ready, y_train)
print("Decision Tree model trained.")


In [ ]:
# 9. Make predictions
y_pred = dt.predict(X_test_ready)


In [ ]:
# 10. Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Survived", "Died"], zero_division=0))


In [ ]:
# 11. Visualize the Decision Tree
plt.figure(figsize=(22, 12))
plot_tree(
    dt,
    feature_names=X.columns,
    class_names=["Survived", "Died"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree for COVID-19 Patient Death Prediction")
plt.show()


In [ ]:
# 12. Feature importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance.head(15))


## Important note
- This is a machine-learning exercise, not a medical prediction tool.
- The notebook assumes `DATE_DIED` uses `9999-99-99` for patients who survived. Check this value in your CSV before running.
- The replacement of 97/98/99 with missing values is a practical preprocessing assumption for this dataset; verify the coding scheme for each column if your course requires strict data cleaning.
- Decision trees do not require feature scaling, so scaling from the Iris notebook is intentionally omitted.